![Health-Informatics: A Python Tutorial](../../Image/chapter-banner.png)


# Module 6:  Data Ingestion and ETL Pipelines



**Health Informatics in Python** · Part II: Interoperability & Data Engineering · Module 6 of 16

---



Real health data arrives messy, in mixed formats and units, from more than one
source. Before anyone can analyze it, an **ETL pipeline** (Extract → Transform →
Load) has to ingest, clean, normalize, and align it. This module builds a small
but realistic, reusable pipeline.


## Learning objectives

By the end of this module you will be able to:

1. **Extract** from more than one source (a flat export and a FHIR bundle).
2. **Transform**: clean, **normalize units**, align timestamps, and handle missingness.
3. **Load** into tidy, analysis-ready tables.
4. Assemble the steps into a **reusable, logged pipeline** with a validation gate.
5. Explain where **orchestration** tools (Airflow, Prefect) fit.


## Dataset

Two deliberately *messy* sources describing the same kind of data: a **CSV export**
with mixed units and missing values, and a small **FHIR-style JSON** feed. The goal
is to reconcile them into one clean table.


In [1]:
# --- Self-contained synthetic EHR generator (identical to Part I) ---
# This is the SAME generator from Module 1. We rebuild the fake hospital database
# here so this notebook can run on its own. Nothing here is a real patient.
import numpy as np
import pandas as pd

# This function generates a synthetic electronic health record (EHR) dataset.
# It creates mock tables for patients, encounters, observations, conditions, and medications,
# all with plausible structure, but not referencing any real patients or requiring external data.
def make_synthetic_ehr(n_patients=200, seed=42):
    """
    Generate a small, internally-consistent synthetic EHR dataset.
    Returns a dict of linked DataFrames: patients, encounters, observations,
    conditions, and medications. This is for illustrative, educational use.
    """
    rng = np.random.default_rng(seed)  # same seed => same fake people every run

    # --- patients table ---
    # Create a DataFrame with n_patients patients, each with a name, sex, age, and birth year.
    first = ["Ava","Liam","Noah","Mia","Zoe","Omar","Ivan","Sara","Leo","Nina",
             "Ruth","Kai","Yara","Theo","Ida","Sam","Ana","Eli","Rex","Uma"]
    last  = ["Khan","Ortiz","Chen","Diaz","Patel","Ali","Brown","Nash","Reed","Vega",
             "Cole","Frost","Grant","Hale","Iqbal","Jain","Kerr","Lund","Mora","Park"]
    sexes = rng.choice(["male", "female"], size=n_patients, p=[0.49, 0.51])
    ages  = rng.integers(18, 90, size=n_patients)
    # Columns:
    #   patient_id  — unique ID, e.g. "P1000", "P1001"
    #   given_name / family_name — randomly chosen from the name lists
    #   sex, age — pre-generated arrays
    #   birth_year — derived from age, assuming the reference year 2026
    patients = pd.DataFrame({
        "patient_id": [f"P{1000+i}" for i in range(n_patients)],
        "given_name": rng.choice(first, size=n_patients),
        "family_name": rng.choice(last, size=n_patients),
        "sex": sexes,
        "age": ages,
        "birth_year": 2026 - ages,
    })

    # --- encounters table ---
    # Each patient gets 1–4 visits. Types and dates are random but internally consistent.
    #   encounter_id    — unique, zero-padded (E00001, E00002, ...)
    #   encounter_type  — ambulatory 55%, emergency 15%, inpatient 10%, wellness 20%
    #   date            — a random day in a 3-year window starting 2023-01-01
    enc_rows = []
    enc_types = ["ambulatory", "emergency", "inpatient", "wellness"]
    for pid in patients["patient_id"]:
        for _ in range(rng.integers(1, 5)):  # 1 to 4 encounters per patient
            day = rng.integers(0, 365*3)     # offset within 3 years
            enc_rows.append({
                "encounter_id": f"E{len(enc_rows)+1:05d}",
                "patient_id": pid,
                "encounter_type": rng.choice(enc_types, p=[0.55, 0.15, 0.10, 0.20]),
                "date": (pd.Timestamp("2023-01-01") + pd.Timedelta(days=int(day))).date(),
            })
    encounters = pd.DataFrame(enc_rows)

    # --- observations table ---
    # Labs and vitals attached to encounters. Each tuple is (name, unit, lo, hi).
    # A value is drawn uniformly in [lo, hi]; 70% of (encounter, measure) pairs exist
    # so the table looks sparse the way a real EHR does.
    obs_defs = [
        ("Body height", "cm", 150, 195),
        ("Body weight", "kg", 50, 110),
        ("Systolic blood pressure", "mmHg", 100, 165),
        ("Heart rate", "/min", 55, 100),
        ("Hemoglobin A1c", "%", 4.8, 9.5),
    ]
    obs_rows = []
    for _, e in encounters.iterrows():
        for name, unit, lo, hi in obs_defs:
            if rng.random() < 0.7:
                obs_rows.append({
                    "observation_id": f"O{len(obs_rows)+1:06d}",
                    "encounter_id": e["encounter_id"],
                    "patient_id": e["patient_id"],
                    "observation": name,
                    "value": round(float(rng.uniform(lo, hi)), 1),
                    "unit": unit,
                    "date": e["date"],
                })
    observations = pd.DataFrame(obs_rows)

    # --- conditions table ---
    # Each patient is assigned 0–3 unique diagnoses from a small pool.
    cond_pool = ["Essential hypertension", "Type 2 diabetes mellitus", "Asthma",
                 "Acute bronchitis", "Major depressive disorder", "Osteoarthritis",
                 "Chronic kidney disease", "Anemia"]
    cond_rows = []
    for pid in patients["patient_id"]:
        for c in rng.choice(cond_pool, size=rng.integers(0, 4), replace=False):
            cond_rows.append({
                "condition_id": f"C{len(cond_rows)+1:05d}",
                "patient_id": pid,
                "condition": c,
            })
    conditions = pd.DataFrame(cond_rows)

    # --- medications table ---
    # Same pattern as conditions: 0–3 unique drugs per patient.
    med_pool = ["Lisinopril", "Metformin", "Albuterol", "Atorvastatin",
                "Sertraline", "Amoxicillin", "Ibuprofen", "Hydrochlorothiazide"]
    med_rows = []
    for pid in patients["patient_id"]:
        for m in rng.choice(med_pool, size=rng.integers(0, 4), replace=False):
            med_rows.append({
                "medication_id": f"M{len(med_rows)+1:05d}",
                "patient_id": pid,
                "medication": m,
            })
    medications = pd.DataFrame(med_rows)

    return {
        "patients": patients,
        "encounters": encounters,
        "observations": observations,
        "conditions": conditions,
        "medications": medications,
    }

# Generate the five linked tables. Each notebook in this series starts from here.
ehr = make_synthetic_ehr()
print("Tables:", ", ".join(f"{k} ({len(v)} rows)" for k, v in ehr.items()))


Tables: patients (200 rows), encounters (511 rows), observations (1804 rows), conditions (304 rows), medications (276 rows)


In [2]:
# SOURCE A is a messy CSV export — the kind a warehouse dump or vendor feed
# actually looks like. Three problems we will have to fix later:
#   1. Mixed units: weight in lb AND kg, height in in AND cm
#   2. Inconsistent labels: "a1c" instead of "Hemoglobin A1c"
#   3. Missing values and mixed date formats (2023-05-02 vs 2023/06/11)
#
# io.StringIO(csv) pretends a string is a file, so pd.read_csv can read it
# without writing anything to disk.
import pandas as pd
import numpy as np
import io
messy_csv = '''patient_id,measure,reading,units,recorded
P1000,weight,187,lb,2023-05-02
P1000,height,70,in,2023-05-02
P1000,a1c,7.1,%,2023-05-02
P1001,weight,82,kg,2023/06/11
P1001,a1c,,%,2023-06-11
P1002,weight,205,lb,2023-07-19
P1002,height,168,cm,2023-07-19'''

source_a = pd.read_csv(io.StringIO(messy_csv))
print("SOURCE A (messy CSV):")
print(source_a.to_string(index=False))


SOURCE A (messy CSV):
patient_id measure  reading units   recorded
     P1000  weight    187.0    lb 2023-05-02
     P1000  height     70.0    in 2023-05-02
     P1000     a1c      7.1     % 2023-05-02
     P1001  weight     82.0    kg 2023/06/11
     P1001     a1c      NaN     % 2023-06-11
     P1002  weight    205.0    lb 2023-07-19
     P1002  height    168.0    cm 2023-07-19


In [3]:
# SOURCE B is a FHIR-style observation feed: already coded (LOINC), already in
# SI units, nested as a list of dicts instead of a flat table.
# Real FHIR would wrap each of these in an Observation resource; we keep the
# payload small so the extract step is easy to follow.
#
# Notice the column names differ from Source A:
#   subject vs patient_id, display vs measure, date vs recorded, ...
# That mismatch is exactly why Extract has to map both sources onto one schema.
source_b = [
    {"subject": "P1003", "loinc": "29463-7", "display": "Body weight",
     "value": 74.0, "unit": "kg", "date": "2023-08-01"},
    {"subject": "P1003", "loinc": "4548-4", "display": "Hemoglobin A1c",
     "value": 6.4, "unit": "%", "date": "2023-08-01"},
    {"subject": "P1004", "loinc": "8302-2", "display": "Body height",
     "value": 181.0, "unit": "cm", "date": "2023-08-03"},
]
print("SOURCE B (FHIR-style JSON):", len(source_b), "observations")


SOURCE B (FHIR-style JSON): 3 observations


## 6.1 Extract - normalize each source to a common schema

In the Extract step, we take columns with different names and formats from each source
and map them into a common structure. This means renaming columns and reformatting where
necessary, so that whether the data comes from a messy CSV file or a FHIR-style JSON feed,
each row will have the same set of columns: (patient_id, measure, value, unit, date).
This standardization is essential so further processing steps can treat all data uniformly.


In [4]:
# The first transform is STRUCTURAL: map each source's idiosyncratic columns
# onto a shared target schema. After this, every row looks the same:
#   patient_id, measure, value, unit, date
# regardless of whether it came from a CSV dump or a FHIR feed.
TARGET_COLS = ["patient_id", "measure", "value", "unit", "date"]

def extract_source_a(df):
    # CSV used "reading"/"units"/"recorded"; rename them to the target names.
    out = df.rename(columns={"reading": "value", "units": "unit", "recorded": "date"})
    # pd.to_numeric turns the empty A1c cell into NaN instead of a blank string.
    # errors="coerce" means "if it isn't a number, become NaN" rather than crash.
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    return out[TARGET_COLS]

def extract_source_b(records):
    # Walk the list of dicts and pick the fields we care about.
    # .lower().replace("body ", "") turns "Body weight" into "weight" so it
    # matches Source A's "weight" / "height" labels.
    rows = [{
        "patient_id": r["subject"],
        "measure": r["display"].lower().replace("body ", ""),
        "value": r["value"],
        "unit": r["unit"],
        "date": r["date"],
    } for r in records]
    return pd.DataFrame(rows)[TARGET_COLS]

a = extract_source_a(source_a)
b = extract_source_b(source_b)

# Harmonize leftover label differences: Source A said "a1c", we want one name.
a["measure"] = a["measure"].replace({"a1c": "hemoglobin a1c"})
b["measure"] = b["measure"].replace({"a1c": "hemoglobin a1c"})

# Stack the two sources into one table. ignore_index=True rebuilds 0..n row numbers.
raw = pd.concat([a, b], ignore_index=True)
print("Extracted + unified schema:")
print(raw.to_string(index=False))


Extracted + unified schema:
patient_id        measure  value unit       date
     P1000         weight  187.0   lb 2023-05-02
     P1000         height   70.0   in 2023-05-02
     P1000 hemoglobin a1c    7.1    % 2023-05-02
     P1001         weight   82.0   kg 2023/06/11
     P1001 hemoglobin a1c    NaN    % 2023-06-11
     P1002         weight  205.0   lb 2023-07-19
     P1002         height  168.0   cm 2023-07-19
     P1003         weight   74.0   kg 2023-08-01
     P1003 hemoglobin a1c    6.4    % 2023-08-01
     P1004         height  181.0   cm 2023-08-03


## 6.2 Transform - unit normalization

Health data often comes in a mix of units (like pounds vs. kilograms, inches vs. centimeters),
which can lead to serious mistakes if not handled—imagine treating a weight in pounds as if it were kilograms.
To avoid such errors, we standardize (normalize) all measurements to SI units (kg, cm, %).


In [5]:
# Mixed units are a classic source of silent, dangerous errors in health data.
# A weight of 187 lb read as kg is a 2.2× error — enough to change a drug dose.
# We convert everything that isn't already SI into SI (kg, cm, %).
def normalize_units(df):
    df = df.copy()  # never mutate the caller's DataFrame in place
    # Lookup: (measure, current_unit) -> (conversion function, new unit)
    conv = {
        ("weight", "lb"): (lambda v: v * 0.453592, "kg"),  # pounds → kilograms
        ("height", "in"): (lambda v: v * 2.54,     "cm"),  # inches → centimetres
    }

    def apply_row(r):
        key = (r["measure"], r["unit"])
        if key in conv:
            fn, new_unit = conv[key]
            return pd.Series([round(fn(r["value"]), 1), new_unit])
        # Already SI (kg, cm, %) — leave value and unit unchanged
        return pd.Series([r["value"], r["unit"]])

    # axis=1 means "call apply_row once per row, not per column"
    df[["value", "unit"]] = df.apply(apply_row, axis=1)
    return df

normalized = normalize_units(raw)
print("After unit normalization (all SI):")
print(normalized.to_string(index=False))


After unit normalization (all SI):
patient_id        measure  value unit       date
     P1000         weight   84.8   kg 2023-05-02
     P1000         height  177.8   cm 2023-05-02
     P1000 hemoglobin a1c    7.1    % 2023-05-02
     P1001         weight   82.0   kg 2023/06/11
     P1001 hemoglobin a1c    NaN    % 2023-06-11
     P1002         weight   93.0   kg 2023-07-19
     P1002         height  168.0   cm 2023-07-19
     P1003         weight   74.0   kg 2023-08-01
     P1003 hemoglobin a1c    6.4    % 2023-08-01
     P1004         height  181.0   cm 2023-08-03


### Milestone 1 - align timestamps and handle missingness

In this step, we standardize the date formats (align timestamps) so all dates are uniform and machine-readable,
and we handle missing data by removing any rows that do not have a measured value,
ensuring only complete, properly-timestamped records are used downstream.


In [6]:
# Two remaining messes: dates in mixed formats, and a row with no measured value.
def clean_temporal_and_missing(df):
    df = df.copy()
    # Source A mixed "2023-05-02" and "2023/06/11". format="mixed" lets pandas
    # try several date layouts; errors="coerce" turns unparseable dates into NaT
    # (Not-a-Time) instead of crashing.
    df["date"] = pd.to_datetime(df["date"], format="mixed", errors="coerce")

    before = len(df)
    # Drop rows with no measured value. We do NOT impute a lab that was never
    # taken — inventing an A1c is worse than leaving the row out.
    df = df.dropna(subset=["value"])
    dropped = before - len(df)
    print(f"Dropped {dropped} row(s) with missing value; {len(df)} remain.")
    return df

clean = clean_temporal_and_missing(normalized)
print(clean.to_string(index=False))


Dropped 1 row(s) with missing value; 9 remain.
patient_id        measure  value unit       date
     P1000         weight   84.8   kg 2023-05-02
     P1000         height  177.8   cm 2023-05-02
     P1000 hemoglobin a1c    7.1    % 2023-05-02
     P1001         weight   82.0   kg 2023-06-11
     P1002         weight   93.0   kg 2023-07-19
     P1002         height  168.0   cm 2023-07-19
     P1003         weight   74.0   kg 2023-08-01
     P1003 hemoglobin a1c    6.4    % 2023-08-01
     P1004         height  181.0   cm 2023-08-03


## 6.3 Load — a validation gate before anything ships

Before loading data into downstream systems, the pipeline must enforce basic data quality checks—rejecting any records that fail plausibility rules. Here, we introduce a validation "gate" that catches obviously erroneous values (such as impossible heights or weights), inspired by practices you'll learn in depth in Module 8.


In [7]:
# A pipeline should REFUSE TO LOAD data that fails basic checks.
# These ranges are coarse "could a human possibly have this value?" bounds,
# not clinical reference ranges. A weight of 500 kg is a data error; 90 kg is not.
# This is a lightweight preview of Module 8's data-quality work.
PLAUSIBLE = {  # measure → (min, max) in SI units
    "weight": (2, 350),
    "height": (30, 250),
    "hemoglobin a1c": (3, 20),
}

def validate(df):
    problems = []
    for _, r in df.iterrows():
        rng = PLAUSIBLE.get(r["measure"])  # None if we have no range for this measure
        if rng and not (rng[0] <= r["value"] <= rng[1]):
            problems.append((r["patient_id"], r["measure"], r["value"]))
    return problems

issues = validate(clean)
print("Validation gate:", "PASS" if not issues else f"FAIL {issues}")


Validation gate: PASS


### Milestone 2 - assemble the reusable pipeline

In this section, we'll assemble all the ETL steps into a single, reusable pipeline function.
This function will combine extraction, transformation, cleaning, and validation,
ensuring that all data passes through the same process before being loaded for analytics.

In [8]:
# Wrap the steps we already wrote into one reusable function with logging.
# In production this is the function an orchestrator (Airflow, Prefect) would call.
#
# logging.basicConfig sets up how messages look:
#   stream=sys.stdout  — print to the notebook, not a hidden file
#   format="[%(levelname)s] %(message)s"  — e.g. [INFO] EXTRACT: reading 2 sources
#   force=True  — replace any logger a previous cell may have configured
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO,
                    format="[%(levelname)s] %(message)s", force=True)
log = logging.getLogger("etl")  # named logger so ETL messages are easy to filter

def run_pipeline(csv_df, fhir_records):
    log.info("EXTRACT: reading 2 sources")
    a = extract_source_a(csv_df)
    b = extract_source_b(fhir_records)
    a["measure"] = a["measure"].replace({"a1c": "hemoglobin a1c"})
    b["measure"] = b["measure"].replace({"a1c": "hemoglobin a1c"})
    raw = pd.concat([a, b], ignore_index=True)

    log.info(f"TRANSFORM: {len(raw)} rows -> normalizing units")
    df = normalize_units(raw)
    df = clean_temporal_and_missing(df)

    log.info("VALIDATE: running plausibility gate")
    if validate(df):
        # Abort rather than load bad data. Module 8 will show a quarantine alternative.
        log.error("LOAD ABORTED: validation failed")
        return None

    log.info(f"LOAD: {len(df)} clean rows ready for analytics")
    return df.sort_values(["patient_id", "date"]).reset_index(drop=True)

analytics_ready = run_pipeline(source_a, source_b)
analytics_ready


[INFO] EXTRACT: reading 2 sources
[INFO] TRANSFORM: 10 rows -> normalizing units
Dropped 1 row(s) with missing value; 9 remain.
[INFO] VALIDATE: running plausibility gate
[INFO] LOAD: 9 clean rows ready for analytics


,patient_id,measure,value,unit,date
0,P1000,weight,84.8,kg,2023-05-02
1,P1000,height,177.8,cm,2023-05-02
2,P1000,hemoglobin a1c,7.1,%,2023-05-02
3,P1001,weight,82.0,kg,2023-06-11
4,P1002,weight,93.0,kg,2023-07-19
5,P1002,height,168.0,cm,2023-07-19
6,P1003,weight,74.0,kg,2023-08-01
7,P1003,hemoglobin a1c,6.4,%,2023-08-01
8,P1004,height,181.0,cm,2023-08-03


## 6.4 Where orchestration fits
 
The pipeline function defined above executes the ETL (Extract, Transform, Load) workflow as a single sequence, running each step in order from start to finish within one call. While this linear execution works well in experimental or prototyping environments like notebooks, real-world data systems require additional capabilities to ensure robust, reliable, and manageable data processing. 

In a production environment, you typically want to introduce **orchestration** — the process of managing, monitoring, and automating the execution of data pipelines. Dedicated orchestration tools such as **Apache Airflow** and **Prefect** provide features like:
- **Scheduling**: Automatically running pipelines at specified times, such as hourly, daily, or in response to external triggers.
- **Retries**: If a step fails, automatically attempting it again without manual intervention, possibly with configurable backoff and thresholds.
- **Dependency Management**: Defining the precise order in which steps (known as “tasks”) should run based on their dependencies; e.g., not starting a transformation until extraction is complete.
- **Monitoring and Alerting**: Keeping track of the status and health of pipeline runs, and sending notifications or logging errors if something goes wrong.

These tools visualize and control your pipeline as a **Directed Acyclic Graph (DAG)**, where each function or step in your process is represented as a task. The conceptual structure for the pipeline above might look like this:
 
```
extract_a ─┐
           ├─> transform ─> validate ─> load
extract_b ─┘
```
 
In this DAG, the `extract_a` and `extract_b` tasks (which can run independently or in parallel) both feed their outputs into a `transform` task. The transformed data then passes through `validate` and `load` tasks sequentially. The DAG structure allows the orchestrator to know exactly which steps depend on others, and thus handle execution order, error handling, and retries automatically.

The benefit of using an orchestrator is that you do not have to change the logic of your ETL steps—these remain as Python functions—but you wrap and schedule them within the orchestrator's workflow definition. This separation allows you to build robust data pipelines that can handle failures gracefully, rerun specific steps, integrate with business calendars, monitor for issues, and alert you when things deviate from expectations. All of this is essential for reliable, production-grade data engineering.


## Exercises

1. Add a **temperature** measure to Source A in °F and extend `normalize_units`
   to convert to °C.
2. Make the pipeline **idempotent**: de-duplicate on
   `(patient_id, measure, date)` so re-running doesn't double-load.
3. Change the validation gate to **quarantine** bad rows (write them aside) instead
   of aborting the whole load, and report how many were quarantined.



## Key takeaways

- ETL is **Extract → Transform → Load**; in health data the transform is dominated
  by **unit normalization, temporal alignment, and missingness**.
- A **validation gate** should stand between transform and load.
- Wrap steps into a **logged, reusable function**; hand *scheduling and reliability*
  to an **orchestrator**.



---
*Next: Module 7 - Common Data Models (OMOP CDM).*
